In [1]:
pip install streamlit nltk sentence_transformers seaborn re

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement re (from versions: none)
ERROR: No matching distribution found for re


In [24]:
import numpy as np
import pandas as pd
import nltk
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.metrics.pairwise import linear_kernel,cosine_similarity
import seaborn as sns
from ast import literal_eval
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [25]:
df = pd.read_csv('archive\TMDB_IMDB_Movies_Dataset.csv')


## Looking at starting rows of both dataframes to get idea what we are dealing with

In [26]:
print("First 5 Data:")
df.head()

First 5 Data:


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc...",Christopher Nolan,Christopher Nolan,8.8,2799547,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan",8.7,2500715,"Matthew McConaughey, Anne Hathaway, Michael Ca..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f...",Christopher Nolan,"Jonathan Nolan, Christopher Nolan, David S. Go...",9.1,3149407,"Christian Bale, Heath Ledger, Aaron Eckhart, M..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ...",James Cameron,James Cameron,7.9,1490771,"Sam Worthington, Zoe Saldaña, Sigourney Weaver..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com...",Joss Whedon,"Joss Whedon, Zak Penn",8.0,1551474,"Robert Downey Jr., Chris Evans, Mark Ruffalo, ..."


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 436204 entries, 0 to 436203
Data columns (total 29 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    436204 non-null  int64  
 1   title                 436204 non-null  object 
 2   vote_average          436204 non-null  float64
 3   vote_count            436204 non-null  int64  
 4   status                436204 non-null  object 
 5   release_date          414939 non-null  object 
 6   revenue               436204 non-null  int64  
 7   runtime               436204 non-null  int64  
 8   adult                 436204 non-null  bool   
 9   backdrop_path         185215 non-null  object 
 10  budget                436204 non-null  int64  
 11  homepage              54751 non-null   object 
 12  tconst                436204 non-null  object 
 13  original_language     436204 non-null  object 
 14  original_title        436204 non-null  object 
 15  

In [28]:
df.describe()

,id,vote_average,vote_count,revenue,runtime,budget,popularity,averageRating,numVotes
count,4.362040e+05,436204.000000,436204.000000,4.362040e+05,436204.000000,4.362040e+05,436204.000000,436204.000000,4.362040e+05
mean,4.930923e+05,3.600559,48.856154,1.652450e+06,66.461461,6.483346e+05,2.194258,6.230138,3.097229e+03
std,3.632846e+05,3.149695,512.721363,2.618312e+07,63.892368,7.411128e+06,11.343042,1.309261,3.394902e+04
min,2.000000e+00,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,1.000000,5.000000e+00
25%,2.099208e+05,0.000000,0.000000,0.000000e+00,15.000000,0.000000e+00,0.600000,5.400000,2.300000e+01
50%,4.189680e+05,4.600000,1.000000,0.000000e+00,80.000000,0.000000e+00,0.855000,6.300000,6.900000e+01
75%,7.102568e+05,6.200000,6.000000,0.000000e+00,96.000000,0.000000e+00,1.658000,7.100000,3.180000e+02
max,1.658262e+06,10.000000,34495.000000,2.923706e+09,14400.000000,8.880000e+08,2994.357000,10.000000,3.170915e+06


In [29]:
df.isnull()
df.isnull().sum()

id                           0
title                        0
vote_average                 0
vote_count                   0
status                       0
release_date             21265
revenue                      0
runtime                      0
adult                        0
backdrop_path           250989
budget                       0
homepage                381453
tconst                       0
original_language            0
original_title               0
overview                 42891
popularity                   0
poster_path              75817
tagline                 344074
genres                   79588
production_companies    174382
production_countries    115587
spoken_languages        104240
keywords                265548
directors                 9938
writers                  65026
averageRating                0
numVotes                     0
cast                     68777
dtype: int64

In [30]:
#since we have no use of these data does not matter for the recommendation system so I drop it
df = df.drop(columns=['homepage', 'backdrop_path', 'poster_path'])

In [31]:
df.isnull().sum()

id                           0
title                        0
vote_average                 0
vote_count                   0
status                       0
release_date             21265
revenue                      0
runtime                      0
adult                        0
budget                       0
tconst                       0
original_language            0
original_title               0
overview                 42891
popularity                   0
tagline                 344074
genres                   79588
production_companies    174382
production_countries    115587
spoken_languages        104240
keywords                265548
directors                 9938
writers                  65026
averageRating                0
numVotes                     0
cast                     68777
dtype: int64

In [32]:
#main feauture will be  = overview + genres + keywords + cast + directors
#checking for text normalization
df['genres'].iloc[0]
df['overview'].iloc[1]

'The adventures of a group of explorers who make use of a newly discovered wormhole to surpass the limitations on human space travel and conquer the vast distances involved in an interstellar voyage.'

In [33]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # lowercase
    text = text.lower()
    
    # remove special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # tokenize
    words = text.split()
    
    # remove stopwords + lemmatize
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return " ".join(words)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lifei\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\lifei\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [34]:
df.isnull().sum()

id                           0
title                        0
vote_average                 0
vote_count                   0
status                       0
release_date             21265
revenue                      0
runtime                      0
adult                        0
budget                       0
tconst                       0
original_language            0
original_title               0
overview                 42891
popularity                   0
tagline                 344074
genres                   79588
production_companies    174382
production_countries    115587
spoken_languages        104240
keywords                265548
directors                 9938
writers                  65026
averageRating                0
numVotes                     0
cast                     68777
dtype: int64

In [35]:
#due to the high amount of data, we would like to use only the top 20000 based on the imdb Weighted Average Ratings
m = df['vote_count'].quantile(0.9)
C = df['vote_average'].mean()

#Defining Formula for calculating the score of each movie
def weight_average(x):
    v = x['vote_count']
    R = x['vote_average']
    return (v/(v+m))*R + (m/(v+m))*C

#Filtering Dataframe to get movies with vote count >= m
q_movies = df.copy().loc[df['vote_count'] >= m]
q_movies.shape


(44568, 26)

In [36]:
#Adding new feature 'score' according to IMDB formula to rank these movies rating-wise
q_movies['score'] = q_movies.apply(weight_average , axis = 1)
q_movies.shape

(44568, 27)

In [37]:
#Sorting According to 'Score' attribute and find top 20 IMDB rated movies
q_movies = q_movies.sort_values('score' , ascending = False)
q_movies[['title','score','vote_average','vote_count']].head(20)

,title,score,vote_average,vote_count
53,The Godfather,8.700447,8.707,18677
14,The Shawshank Redemption,8.697038,8.702,24649
215,The Godfather Part II,8.580417,8.591,11293
119,Schindler's List,8.564836,8.573,14594
107,Spirited Away,8.531065,8.539,14913
436,12 Angry Men,8.524568,8.540,7658
991,Dilwale Dulhania Le Jayenge,8.524235,8.552,4256
2,The Dark Knight,8.508153,8.512,30619
81,Parasite,8.507832,8.515,16430
258,Your Name.,8.502581,8.514,10303


In [38]:
#Keep top 20,000
df_model = q_movies.sort_values('score', ascending=False).head(20000).copy()

# remove duplicate titles
df_model = df_model.drop_duplicates(subset='title').reset_index(drop=True)

In [39]:
df_model.isnull().sum()

id                         0
title                      0
vote_average               0
vote_count                 0
status                     0
release_date               0
revenue                    0
runtime                    0
adult                      0
budget                     0
tconst                     0
original_language          0
original_title             0
overview                  52
popularity                 0
tagline                 7413
genres                    22
production_companies     644
production_countries     199
spoken_languages          58
keywords                2403
directors                 47
writers                  668
averageRating              0
numVotes                   0
cast                     197
score                      0
dtype: int64

In [40]:
# Heavy cleaning for text fields
for feature in ['overview']:
    df_model[feature] = df_model[feature].fillna('').apply(clean_text)

# Light cleaning for categorical text
for feature in ['genres', 'keywords']:
    df_model[feature] = df_model[feature].fillna('').str.lower()

# Special handling for names
for feature in ['cast', 'directors']:
    df_model[feature] = df_model[feature].fillna('').str.lower()
    df_model[feature] = df_model[feature].str.replace(' ', '', regex=False)

In [41]:
df_model.isnull().sum()

id                         0
title                      0
vote_average               0
vote_count                 0
status                     0
release_date               0
revenue                    0
runtime                    0
adult                      0
budget                     0
tconst                     0
original_language          0
original_title             0
overview                   0
popularity                 0
tagline                 7413
genres                     0
production_companies     644
production_countries     199
spoken_languages          58
keywords                   0
directors                  0
writers                  668
averageRating              0
numVotes                   0
cast                       0
score                      0
dtype: int64

In [42]:
df_model['soup'] = (
    df_model['overview'] + ' ' +
    df_model['genres'] + ' ' +
    df_model['keywords'] + ' ' +
    df_model['cast'] + ' ' +
    df_model['directors']
)

In [1]:
#df_model['soup'].tolist()

In [43]:
# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(
    df_model['soup'].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/597 [00:00<?, ?it/s]

tensor([[1.0000, 0.3635, 0.8290,  ..., 0.2939, 0.3147, 0.2225],
        [0.3635, 1.0000, 0.2855,  ..., 0.1721, 0.2317, 0.3768],
        [0.8290, 0.2855, 1.0000,  ..., 0.3179, 0.3031, 0.2538],
        ...,
        [0.2939, 0.1721, 0.3179,  ..., 1.0000, 0.2375, 0.1927],
        [0.3147, 0.2317, 0.3031,  ..., 0.2375, 1.0000, 0.1413],
        [0.2225, 0.3768, 0.2538,  ..., 0.1927, 0.1413, 1.0000]])


In [55]:
indices = pd.Series(df_model.index, index=df_model['title']).drop_duplicates()
def get_recommendations(title, similarities, df_model, top_n=10):
    idx = indices[title]
    sim_scores = list(enumerate(similarities[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]

    movie_indices = [i[0] for i in sim_scores]
    scores = [float(i[1]) for i in sim_scores]

    result = df_model.iloc[movie_indices][['title']].copy()
    result['similarity_score'] = scores
    return result

In [56]:
get_recommendations("Spirited Away", similarities, df_model)

,title,similarity_score
2446,Puella Magi Madoka Magica the Movie Part III: ...,0.670731
63,Hotarubi no Mori e,0.670384
441,Suzume,0.667450
302,The Boy and the Beast,0.667347
5442,Puella Magi Madoka Magica the Movie Part I: Be...,0.667032
5331,Puella Magi Madoka Magica the Movie Part II: E...,0.666351
3574,Mary and The Witch's Flower,0.663136
32,Howl's Moving Castle,0.646826
15005,The Promised Neverland,0.642051
274,Weathering with You,0.624517


In [57]:
df_model[df_model['title'] == "Spirited Away"][['title', 'overview', 'genres', 'keywords', 'cast', 'directors', 'soup']]

,title,overview,genres,keywords,cast,directors,soup
4,Spirited Away,young girl chihiro becomes trapped strange new...,"animation, family, fantasy","witch, parent child relationship, darkness, ba...","rumihiiragi,miyuirino,marinatsuki,takashinaito...",hayaomiyazaki,young girl chihiro becomes trapped strange new...
